In [6]:
import xisf
import cv2
import numpy as np
import os
import csv
import random
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from astropy.stats import sigma_clipped_stats
from astropy.visualization import AsinhStretch, ImageNormalize, ManualInterval, ZScaleInterval
from photutils.detection import DAOStarFinder
import traceback
import tkinter as tk
from tkinter import filedialog

# --- Configuration ---
# Set this to the pattern that looked correct in the Diagnostic (e.g., 'BGGR')
# Set to None to use the file's metadata default.
FORCE_BAYER_PATTERN = 'BGGR' # User confirmed BGGR looks correct (Red Orion) vs RGGB (Blue Orion). Metadata likely inverted.

# Fallback if FORCE is None and Metadata is missing
DEFAULT_BAYER_PATTERN = 'RGGB' 

# Star Extraction Settings (From project.ipynb)
# OPTIMIZED: High threshold for noise, but loose roundness to catch tracking errors
FWHM_PIXELS = 4.0
THRESHOLD_SIGMA = 8.0 # High threshold to avoid noise/hot pixels
CUTOUT_SIZE = 256
MAX_STARS_PER_IMAGE = 50 
SHARPLO = 0.2
SHARPHI = 0.9
ROUNDLO = -1.0 # Reverted to -1.0 to include tracking errors (elongated stars)
ROUNDHI = 1.0  # Reverted to 1.0

# Output Definitions
XISF_CODEC = 'zlib'
XISF_SHUFFLE = True

bayer_pattern_map = {
    'RGGB': cv2.COLOR_BAYER_RG2RGB,
    'GRBG': cv2.COLOR_BAYER_GR2RGB,
    'GBRG': cv2.COLOR_BAYER_GB2RGB, 
    'BGGR': cv2.COLOR_BAYER_BG2RGB,
}

In [7]:
def select_data_directory():
    root = tk.Tk()
    root.withdraw()
    print("Please select your main 'data' directory...")
    dir_path = filedialog.askdirectory(title="Select your 'data' directory (containing 'original_data')")
    return Path(dir_path) if dir_path else None

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def load_xisf_linear_color_and_gray(file_path: Path):
    # Load Debayered Image
    f = xisf.XISF(str(file_path))
    image_obj = f.read_image(0)
    data_color = np.asarray(image_obj.data, dtype=np.float32)
    
    if data_color.ndim != 3 or data_color.shape[2] != 3:
        # If not 3 channel (already gray?), handle or error
        if data_color.ndim == 2:
             return np.stack([data_color]*3, axis=-1), data_color
        raise ValueError(f"Expected RGB image: {file_path}")

    # Create Grayscale for detection
    data_gray = (
        0.299 * data_color[:, :, 0] + 
        0.587 * data_color[:, :, 1] + 
        0.114 * data_color[:, :, 2]
    )
    return data_color, data_gray

def process_file_debayer(args):
    file_path, base_input_dir, base_output_debayered = args
    try:
        relative_path = file_path.relative_to(base_input_dir)
        output_dir = base_output_debayered / relative_path.parent
        ensure_dir(output_dir)
        
        output_path = output_dir / f"{file_path.stem}_debayered.xisf"
        
        # Check exists
        if output_path.exists():
            return None # Skip if exists
            
        # Load Image AND Metadata
        image_metadata = {}
        image_obj = xisf.XISF.read(str(file_path), image_metadata=image_metadata)
        
        # Extract Bayer Pattern
        if FORCE_BAYER_PATTERN:
            bayer_pattern = FORCE_BAYER_PATTERN
        else:
            bayer_pattern = DEFAULT_BAYER_PATTERN
            fits_keywords = image_metadata.get('FITSKeywords', {})
            if fits_keywords:
                bayerpat_info = fits_keywords.get('BAYERPAT', [])
                if bayerpat_info and isinstance(bayerpat_info, list) and len(bayerpat_info) > 0:
                    bayer_pattern = bayerpat_info[0].get('value', DEFAULT_BAYER_PATTERN).strip().upper()
                    bayer_pattern = bayer_pattern.replace("'", "").replace('"', '')

        # Debayer
        debayer_code = bayer_pattern_map.get(bayer_pattern)
        if debayer_code is None:
            return f"Error: Unknown/Unsupported pattern '{bayer_pattern}' in {file_path.name}"
            
        rgb_image = cv2.cvtColor(image_obj, debayer_code)
        
        # --- BACKGROUND NEUTRALIZATION (New: Matching Project Notebook) ---
        # Raw debayered images are very green. Project notebook neutralized this.
        height, width = rgb_image.shape[:2]
        small_rgb = cv2.resize(rgb_image, (width // 16, height // 16), interpolation=cv2.INTER_AREA)
        r_chan_small, g_chan_small, b_chan_small = cv2.split(small_rgb)
        r_med = np.median(r_chan_small)
        g_med = np.median(g_chan_small)
        b_med = np.median(b_chan_small)

        rgb_balanced_image = rgb_image
        epsilon = 1e-6
        if r_med > epsilon and g_med > epsilon and b_med > epsilon:
            r_scale = np.float32(g_med / r_med)
            b_scale = np.float32(g_med / b_med)
            
            r_chan, g_chan, b_chan = cv2.split(rgb_image)
            r_balanced = (r_chan.astype(np.float32) * r_scale)
            b_balanced = (b_chan.astype(np.float32) * b_scale)
            
            rgb_balanced_float = cv2.merge([r_balanced, g_chan.astype(np.float32), b_balanced])
            
            # SCALING FIX: Ensure range is 0-65535
            # If input was float 0-1, we must scale up before casting.
            if rgb_balanced_float.max() <= 1.0:
                 rgb_balanced_float = rgb_balanced_float * 65535.0

            # Clip to valid range
            rgb_balanced_image = np.clip(rgb_balanced_float, 0, 65535).astype(np.uint16)
        else:
             # Avoid divide by zero, fallback to raw
             pass

        # Cleanup Metadata (Remove BAYERPAT since it is now RGB)
        if 'FITSKeywords' in image_metadata:
             # Filter out BAYERPAT
             # Note: FITSKeywords values are lists of dicts usually?
             # Here we just want to ensure we don't save a misleading header if possible.
             # However, modifying complex nested dicts safely is tricky without breaking xisf writer if it expects strict structure.
             # Simplest fix: The file is now RGB, xisf writer handles dimensions.
             # We'll leave metadata as is (history) to avoid write errors, matching project notebook behavior.
             pass

        # Save Debayered (Balanced)
        xisf.XISF.write(str(output_path), rgb_balanced_image, image_metadata=image_metadata, codec=XISF_CODEC, shuffle=XISF_SHUFFLE)
        return f"Debayered (Balanced): {file_path.name} (Pattern: {bayer_pattern})"
        
    except Exception as e:
        traceback.print_exc()
        return f"Error debayering {file_path.name}: {e}"

def extract_stars_and_save(args):
    file_path, base_output_debayered, base_output_cutouts = args
    stars_in_image = [] 
    log_msgs = []
    
    try:
        # Determine Class from Folder Name
        # Assuming structure: .../debayered_output/ClassName/ImageName_debayered.xisf
        relative_path = file_path.relative_to(base_output_debayered)
        class_name = relative_path.parent.name
        original_filename = file_path.stem.replace('_debayered', '') + ".xisf" # Approximation

        # Output Directory for this Class
        output_dir = base_output_cutouts / class_name
        ensure_dir(output_dir)

        # Load Data
        data_color, data_gray = load_xisf_linear_color_and_gray(file_path)
        h, w = data_gray.shape
        
        # Background Stats
        mean, median, std = sigma_clipped_stats(data_gray, sigma=3.0, maxiters=5)
        threshold = THRESHOLD_SIGMA * std
        
        # Find Stars
        daofind = DAOStarFinder(fwhm=FWHM_PIXELS, threshold=threshold, sharplo=SHARPLO, sharphi=SHARPHI, roundlo=ROUNDLO, roundhi=ROUNDHI)
        sources = daofind(data_gray - median)
        
        if sources is None or len(sources) == 0:
            return [], [f"No stars in {file_path.name}"]
            
        sources.sort('peak')
        sources.reverse()
        
        valid_stars = 0
        half_size = CUTOUT_SIZE // 2
        
        for i, star in enumerate(sources):
            if valid_stars >= MAX_STARS_PER_IMAGE:
                break
                
            x, y = int(round(star['xcentroid'])), int(round(star['ycentroid']))
            
            # Edge Check
            if x < half_size or x > w - half_size or y < half_size or y > h - half_size:
                continue
                
            # Extract Cutout (All Channels)
            cutout = data_color[y-half_size:y+half_size, x-half_size:x+half_size, :]
            
            # Save Individual Channels (R, G, B) - Linear
            channels = ['R', 'G', 'B']
            for ch_idx, ch_name in enumerate(channels):
                ch_data = cutout[:, :, ch_idx]
                
                # FIX: Ensure we save as standard 16-bit Integer for compatibility.
                # If data is float 0-1, scale up. If float 0-65535, just cast.
                # Use a heuristic relative to the max value of the cutout buffer (or known range)
                if ch_data.max() <= 1.0:
                    ch_data = ch_data * 65535.0
                
                ch_data_uint16 = np.clip(ch_data, 0, 65535).astype(np.uint16)

                # Add singleton dimension for XISF (Height, Width, 1)
                ch_data_3d = ch_data_uint16[..., np.newaxis]
                
                # Filename: OriginalName_star_XX_x_y_Channel_linear.xisf
                # Note: Star number here is just detection order, but CSV will be sorted by X
                out_name = f"{file_path.stem.replace('_debayered','')}_star_{valid_stars+1:02d}_x{x}_y{y}_{ch_name}_linear.xisf"
                save_path = output_dir / out_name
                
                xisf.XISF.write(str(save_path), ch_data_3d, codec=XISF_CODEC, shuffle=XISF_SHUFFLE)
            
            # Add Valid Star to list
            stars_in_image.append((x, y))
            valid_stars += 1
            
        # Sort found stars by X (Left to Right)
        stars_in_image.sort(key=lambda s: s[0])

        log_msgs.append(f"Extracted {valid_stars} stars from {file_path.name}")
        
        # Return single record for this image: (Filename, Class, [List of (x,y)])
        return [(original_filename, class_name, stars_in_image)], log_msgs
        
    except Exception as e:
        traceback.print_exc()
        return [], [f"Error extracting stars {file_path.name}: {e}"]


In [8]:
# --- ANALYSIS TOOL: Star Quantity Analyzer ---
# Run this cell to determine how many stars are available in your images.
# It processes a random sample of files and counts how many stars meet your criteria.
# Use the results to update MAX_STARS_PER_IMAGE at the top.
"""
CONFIG_RUN_ANALYSIS = True
SAMPLE_SIZE = 5

if CONFIG_RUN_ANALYSIS:
    # We need data_dir defined. It will be defined in the Setup Cell below if you run it first,
    # BUT since this tool might be run independently, we might need to ask for it here if not set.
    # For simplicity, we'll assume the user runs the Setup cell below first, or we call select_data_directory localy
    try:
        if 'data_dir' not in locals() or data_dir is None: 
            data_dir = select_data_directory()
    except NameError:
         data_dir = select_data_directory()

    if data_dir:
        base_input = data_dir / 'original_data'
        base_debayered = data_dir / 'final_debayered'
        target_files = sorted(list(base_debayered.rglob('*_debayered.xisf')))
        
        if len(target_files) == 0:
            print("No debayered files found in 'final_debayered'. Please run the 'Step 1: Batch Debayering' cell first.")
        else:
            if len(target_files) > SAMPLE_SIZE:
                sample_files = random.sample(target_files, SAMPLE_SIZE)
            else:
                sample_files = target_files
                
            print(f"--- Running Star Analysis on {len(sample_files)} random images ---")
            print(f"Criteria: FWHM={FWHM_PIXELS}, Threshold={THRESHOLD_SIGMA}sigma, Cutout={CUTOUT_SIZE}px")
            
            counts = []
            
            for fpath in sample_files:
                try:
                    data_color, data_gray = load_xisf_linear_color_and_gray(fpath)
                    h, w = data_gray.shape
                    
                    mean, median, std = sigma_clipped_stats(data_gray, sigma=3.0, maxiters=5)
                    threshold = THRESHOLD_SIGMA * std
                    
                    daofind = DAOStarFinder(fwhm=FWHM_PIXELS, threshold=threshold, sharplo=SHARPLO, sharphi=SHARPHI, roundlo=ROUNDLO, roundhi=ROUNDHI)
                    sources = daofind(data_gray - median)
                    
                    count = 0
                    half_size = CUTOUT_SIZE // 2
                    if sources:
                        for star in sources:
                            x, y = int(round(star['xcentroid'])), int(round(star['ycentroid']))
                            if x >= half_size and x <= w - half_size and y >= half_size and y <= h - half_size:
                                count += 1
                    
                    print(f"  {fpath.name}: {count} candidate stars found.")
                    counts.append(count)
                    
                except Exception as e:
                    print(f"  Error analyzing {fpath.name}: {e}")
            
            if counts:
                avg_stars = np.mean(counts)
                med_stars = np.median(counts)
                min_stars = np.min(counts)
                print(f"\n--- Results ---")
                print(f"Min: {min_stars}, Avg: {avg_stars:.1f}, Median: {med_stars:.1f}")
                print(f"Current Limit: {MAX_STARS_PER_IMAGE}")
                if med_stars > MAX_STARS_PER_IMAGE * 2:
                     print(f"Recommendation: You could increase MAX_STARS_PER_IMAGE to {int(med_stars // 1.5)} safely.")
                elif med_stars < MAX_STARS_PER_IMAGE:
                     print(f"Recommendation: Your limit might be too high. Maybe set to {int(med_stars)}?")
                else:
                     print(f"Recommendation: Your limit seems reasonable.")
    else:
         print("No directory selected.")"""

'\nCONFIG_RUN_ANALYSIS = True\nSAMPLE_SIZE = 5\n\nif CONFIG_RUN_ANALYSIS:\n    # We need data_dir defined. It will be defined in the Setup Cell below if you run it first,\n    # BUT since this tool might be run independently, we might need to ask for it here if not set.\n    # For simplicity, we\'ll assume the user runs the Setup cell below first, or we call select_data_directory localy\n    try:\n        if \'data_dir\' not in locals() or data_dir is None: \n            data_dir = select_data_directory()\n    except NameError:\n         data_dir = select_data_directory()\n\n    if data_dir:\n        base_input = data_dir / \'original_data\'\n        base_debayered = data_dir / \'final_debayered\'\n        target_files = sorted(list(base_debayered.rglob(\'*_debayered.xisf\')))\n\n        if len(target_files) == 0:\n            print("No debayered files found in \'final_debayered\'. Please run the \'Step 1: Batch Debayering\' cell first.")\n        else:\n            if len(target_files

In [9]:
# --- PROCESSING SELECTION & SETUP ---
# Run this cell first to select the dataset and setup paths.

data_dir = select_data_directory()
if data_dir:
    base_input = data_dir / 'original_data'
    base_debayered = data_dir / 'final_debayered'
    base_cutouts = data_dir / 'final_star_cutouts'
    
    ensure_dir(base_debayered)
    ensure_dir(base_cutouts)
    
    # --- Worker Count ---
    # Adjusted to user preference (CPU / 1.25)
    num_workers = max(1, int(os.cpu_count() // 2.5))
    print(f"Setup Complete. Output directories prepared.")
    print(f"Debayer Input: {base_input}")
    print(f"Worker Processes: {num_workers}")
else:
    print("No directory selected.")

Please select your main 'data' directory...
Setup Complete. Output directories prepared.
Debayer Input: /Storage/Files/practicalML/gitlab/practicalml/data/original_data
Worker Processes: 12


In [ ]:
# --- STEP 1: BATCH DEBAYERING ---
# This step converts original .xisf files to color .xisf files.

if 'data_dir' in locals() and data_dir:
    files = list(base_input.rglob('*.xisf'))
    print(f"Found {len(files)} original files for debayering.")
    
    debayer_tasks = [(f, base_input, base_debayered) for f in files]
    
    print(f"Starting Debayering (Pattern: {FORCE_BAYER_PATTERN if FORCE_BAYER_PATTERN else 'Metadata'})...")
    # I reverted the forceful overwrite check comment because previous files are likely fine now.
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        for res in executor.map(process_file_debayer, debayer_tasks):
            if res: print(res)
    print("Debayering Step Complete.")
else:
    print("Please run the 'Setup' cell above first.")

In [22]:
# --- STEP 2: STAR EXTRACTION & CSV GENERATION ---
# This step reads the debayered files, finds stars, saves linear cutouts, and writes the CSV.

num_workers = max(1, int(os.cpu_count() // 1.75))
if 'data_dir' in locals() and data_dir:
    debayered_files = list(base_debayered.rglob('*_debayered.xisf'))
    print(f"Found {len(debayered_files)} debayered files for extraction.")
    
    extract_tasks = [(f, base_debayered, base_cutouts) for f in debayered_files]
    all_csv_rows = []
    
    print("Starting Star Extraction...")
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        results = executor.map(extract_stars_and_save, extract_tasks)
        for csv_rows, logs in results:
            all_csv_rows.extend(csv_rows)
            for log in logs:
                print(log)
    
    # Save CSV
    csv_path = base_cutouts / 'star_locations.csv'
    print(f"Saving CSV to {csv_path}...")
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Original Filename', 'Class', 'Star Coordinates (Left to Right)']) 
        
        for row in all_csv_rows:
            filename, cls, stars = row
            # Format tuples as strings "(x,y)"
            star_strings = [f"({x},{y})" for x, y in stars]
            # Write row with variable length columns
            writer.writerow([filename, cls] + star_strings)
        
    print("Star Extraction Step Complete and CSV Saved!")
else:
    print("Please run the 'Setup' cell above first.")

Found 851 debayered files for extraction.
Starting Star Extraction...


Extracted 50 stars from 2025-06-10_02-51-25_North America Nebula_-0.00_300.00s_0041_debayered.xisf
Extracted 50 stars from 2024-10-12_04-02-23__-0.00_300.00s_0008_debayered.xisf
Extracted 50 stars from 2024-10-12_01-46-15__-0.00_300.00s_0055_debayered.xisf
Extracted 50 stars from 2024-10-21_06-02-36_Great Orion Nebula_-0.00_30.00s_0404_debayered.xisf
Extracted 50 stars from 2024-10-21_01-12-51_M 31_-0.00_300.00s_0269_debayered.xisf
Extracted 50 stars from 2025-06-10_23-54-00_North America Nebula_-0.00_300.00s_0009_debayered.xisf
Extracted 50 stars from 2025-06-10_04-04-54_North America Nebula_-0.00_300.00s_0054_debayered.xisf
Extracted 50 stars from 2024-10-21_01-52-03_M 31_-0.00_300.00s_0276_debayered.xisf
Extracted 26 stars from 2024-10-12_02-10-22__0.00_300.00s_0059_debayered.xisf
Extracted 50 stars from 2025-06-10_02-36-56_North America Nebula_-0.00_300.00s_0039_debayered.xisf
Extracted 50 stars from 2024-10-12_00-59-53__-0.00_300.00s_0048_debayered.xisf
Extracted 28 stars from 202

Extracted 30 stars from 2024-10-20_05-17-26_Great Orion Nebula_0.00_300.00s_0061_debayered.xisf
Extracted 50 stars from 2024-10-13_20-45-29__0.00_300.00s_0072_debayered.xisf
Extracted 50 stars from 2024-10-13_21-12-04__-0.00_300.00s_0077_debayered.xisf
Extracted 50 stars from 2024-10-21_02-04-55_M 31_-0.00_300.00s_0278_debayered.xisf
Extracted 50 stars from 2024-10-21_01-22-56_M 31_0.00_300.00s_0271_debayered.xisf
Extracted 50 stars from 2025-06-11_00-31-10_North America Nebula_-0.00_300.00s_0014_debayered.xisf
Extracted 50 stars from 2024-10-20_00-14-39_M 31_-0.00_300.00s_0191_debayered.xisf
Extracted 50 stars from 2024-10-21_05-37-00_Great Orion Nebula_-0.00_30.00s_0386_debayered.xisf
Extracted 50 stars from 2024-10-20_02-59-48_Great Orion Nebula_-0.00_30.00s_0202_debayered.xisf
No stars in 2024-10-11_23-32-30__-0.00_300.00s_0034_debayered.xisf
Extracted 50 stars from 2024-10-21_05-46-46_Great Orion Nebula_-0.00_30.00s_0395_debayered.xisf
Extracted 50 stars from 2025-06-11_04-19-38_N

Extracted 50 stars from 2024-10-12_03-02-28__-0.00_30.00s_0006_debayered.xisf
Extracted 50 stars from 2025-06-11_02-15-02_North America Nebula_0.00_300.00s_0032_debayered.xisf
Extracted 50 stars from 2024-10-21_05-54-53_Great Orion Nebula_-0.00_30.00s_0399_debayered.xisf
Extracted 50 stars from 2024-10-20_03-00-22_Great Orion Nebula_-0.00_30.00s_0203_debayered.xisf
Extracted 50 stars from 2024-10-20_06-07-45_Great Orion Nebula_-0.00_30.00s_0314_debayered.xisf
Extracted 50 stars from 2024-10-12_00-38-00__-0.00_300.00s_0045_debayered.xisf
Extracted 50 stars from 2024-10-20_06-08-16_Great Orion Nebula_-0.00_30.00s_0315_debayered.xisf
Extracted 50 stars from 2025-06-11_02-05-29_North America Nebula_-0.00_300.00s_0031_debayered.xisf
No stars in 2024-10-12_02-31-12__-0.00_300.00s_0063_debayered.xisf
Extracted 50 stars from 2025-06-10_23-28-09_North America Nebula_-0.00_300.00s_0006_debayered.xisf
Extracted 50 stars from 2024-10-12_02-58-35__-0.00_30.00s_0000_debayered.xisf
Extracted 50 stars

Extracted 50 stars from 2024-10-21_01-30-47_M 31_-0.00_300.00s_0272_debayered.xisf
Extracted 50 stars from 2024-10-11_20-14-26__-0.00_300.00s_0001_debayered.xisf
Extracted 50 stars from 2024-10-20_05-43-51_Great Orion Nebula_-0.00_30.00s_0297_debayered.xisf
No stars in 2024-10-12_02-26-10__-0.00_300.00s_0062_debayered.xisf
Extracted 50 stars from 2024-10-21_05-46-15_Great Orion Nebula_-0.00_30.00s_0394_debayered.xisf
Extracted 32 stars from 2024-10-20_05-22-27_Great Orion Nebula_0.00_300.00s_0062_debayered.xisf


...

Extracted 50 stars from 2025-06-11_03-03-03_North America Nebula_0.00_300.00s_0039_debayered.xisf
Extracted 50 stars from 2024-10-21_00-28-13_M 31_0.00_300.00s_0262_debayered.xisf
No stars in 2024-10-20_19-16-39_M 31_-0.00_300.00s_0209_debayered.xisf
Extracted 50 stars from 2024-10-21_00-54-52_M 31_0.00_300.00s_0266_debayered.xisf
Extracted 50 stars from 2024-10-20_01-05-24_M 31_-0.00_300.00s_0199_debayered.xisf
Extracted 50 stars from 2024-10-20_05-36-14_Great Orion Nebula_-0.00_30.00s_0292_debayered.xisf
Extracted 50 stars from 2024-10-21_01-47-02_M 31_0.00_300.00s_0275_debayered.xisf
Extracted 50 stars from 2024-10-20_06-02-08_Great Orion Nebula_-0.00_30.00s_0310_debayered.xisf
Extracted 2 stars from 2024-10-20_06-22-51_Great Orion Nebula_-0.00_300.00s_0065_debayered.xisf
Extracted 50 stars from 2024-10-12_03-07-08__-0.00_30.00s_0009_debayered.xisf
Extracted 50 stars from 2024-10-20_06-04-46_Great Orion Nebula_-0.00_30.00s_0312_debayered.xisf
Extracted 50 stars from 2025-06-10_23-40

Extracted 50 stars from 2025-06-11_03-31-51_North America Nebula_0.00_300.00s_0043_debayered.xisf
Extracted 50 stars from 2025-06-11_00-36-14_North America Nebula_-0.00_300.00s_0015_debayered.xisf
Extracted 50 stars from 2024-10-21_06-07-53_Great Orion Nebula_-0.00_30.00s_0409_debayered.xisf
Extracted 50 stars from 2024-10-12_03-34-10__-0.00_300.00s_0004_debayered.xisf
No stars in 2024-10-11_23-50-07__0.00_300.00s_0037_debayered.xisf
Extracted 50 stars from 2024-10-20_05-43-18_Great Orion Nebula_-0.00_30.00s_0296_debayered.xisf
Extracted 50 stars from 2025-06-11_03-17-52_North America Nebula_-0.00_300.00s_0041_debayered.xisf
Extracted 50 stars from 2024-10-21_00-18-10_M 31_-0.00_300.00s_0260_debayered.xisf
Extracted 50 stars from 2024-10-12_04-15-29__-0.00_30.00s_0021_debayered.xisf
Extracted 43 stars from 2024-10-20_05-07-21_Great Orion Nebula_0.00_300.00s_0059_debayered.xisf
Extracted 50 stars from 2024-10-21_05-40-13_Great Orion Nebula_-0.00_30.00s_0390_debayered.xisf
Extracted 50 s

Extracted 50 stars from 2024-10-12_03-07-42__0.00_300.00s_0000_debayered.xisf
Extracted 50 stars from 2024-10-21_01-17-55_M 31_-0.00_300.00s_0270_debayered.xisf
Extracted 50 stars from 2024-10-12_00-27-55__-0.00_300.00s_0043_debayered.xisf
Extracted 3 stars from 2024-10-12_00-15-52__0.00_300.00s_0041_debayered.xisf
Extracted 50 stars from 2024-10-12_03-40-26__-0.00_30.00s_0011_debayered.xisf
Extracted 50 stars from 2025-06-11_00-22-56_North America Nebula_0.00_300.00s_0013_debayered.xisf
Extracted 50 stars from 2024-10-20_05-54-10_Great Orion Nebula_-0.00_30.00s_0304_debayered.xisf
Extracted 50 stars from 2024-10-21_00-23-11_M 31_-0.00_300.00s_0261_debayered.xisf
No stars in 2024-10-12_02-21-09__-0.00_300.00s_0061_debayered.xisf
Extracted 50 stars from 2025-06-10_23-48-56_North America Nebula_-0.00_300.00s_0008_debayered.xisf
Extracted 50 stars from 2024-10-20_06-12-08_Great Orion Nebula_-0.00_30.00s_0317_debayered.xisf
Extracted 50 stars from 2024-10-20_02-55-13_Great Orion Nebula_0.0

Extracted 50 stars from 2025-06-11_02-31-19_North America Nebula_-0.00_300.00s_0035_debayered.xisf
No stars in 2024-10-11_23-27-29__0.00_300.00s_0033_debayered.xisf
Extracted 50 stars from 2024-10-21_05-55-27_Great Orion Nebula_-0.00_30.00s_0400_debayered.xisf
Extracted 50 stars from 2025-06-11_04-00-14_North America Nebula_-0.00_300.00s_0047_debayered.xisf
Extracted 50 stars from 2025-06-11_04-29-19_North America Nebula_0.00_300.00s_0050_debayered.xisf
Extracted 50 stars from 2024-10-20_05-41-22_Great Orion Nebula_-0.00_30.00s_0294_debayered.xisf
Extracted 50 stars from 2024-10-20_02-01-42_M 31_-0.00_300.00s_0208_debayered.xisf
Extracted 50 stars from 2024-10-20_01-48-39_M 31_0.00_300.00s_0206_debayered.xisf
Extracted 50 stars from 2025-06-10_02-46-21_North America Nebula_0.00_300.00s_0040_debayered.xisf
Extracted 50 stars from 2024-10-20_00-51-23_M 31_-0.00_300.00s_0197_debayered.xisf
Extracted 50 stars from 2024-10-20_03-02-01_Great Orion Nebula_-0.00_30.00s_0204_debayered.xisf
Extr

Extracted 50 stars from 2024-10-21_05-33-24_Great Orion Nebula_0.00_30.00s_0384_debayered.xisf
Extracted 50 stars from 2024-10-20_05-56-06_Great Orion Nebula_-0.00_30.00s_0306_debayered.xisf
Extracted 50 stars from 2024-10-21_00-45-56_M 31_-0.00_300.00s_0265_debayered.xisf
Extracted 50 stars from 2025-06-11_02-49-13_North America Nebula_-0.00_300.00s_0037_debayered.xisf
Extracted 50 stars from 2024-10-20_01-43-37_M 31_0.00_300.00s_0205_debayered.xisf
Extracted 50 stars from 2025-06-11_02-20-06_North America Nebula_-0.00_300.00s_0033_debayered.xisf
Extracted 50 stars from 2024-10-12_00-53-22__0.00_300.00s_0047_debayered.xisf
Extracted 50 stars from 2024-10-20_05-48-18_Great Orion Nebula_-0.00_30.00s_0300_debayered.xisf
Extracted 40 stars from 2025-06-10_05-13-06_North America Nebula_0.00_300.00s_0061_debayered.xisf
No stars in 2025-06-10_05-23-10_North America Nebula_-0.00_300.00s_0063_debayered.xisf
Extracted 50 stars from 2025-06-10_05-08-04_North America Nebula_-0.00_300.00s_0060_deb

In [ ]:
def convert_xisf_to_jpg(args):
    xisf_path = args
    try:
        jpg_path = xisf_path.with_suffix('.jpg')
        
        # Skip if already exists
        if jpg_path.exists():
            return f"Skipped {xisf_path.name} (JPG exists)"

        # Load XISF
        f = xisf.XISF(str(xisf_path))
        image_obj = f.read_image(0)
        data = np.asarray(image_obj.data, dtype=np.float32)

        # Handle Channels (H, W, C) or (C, H, W) or (H, W)
        if data.ndim == 3 and data.shape[0] in [1, 3]: # (C, H, W)
            data = data.transpose((1, 2, 0))
        
        # Normalize to 0-255 uint8
        if data.max() > 1.0:
             data_norm = (data / 65535.0 * 255.0)
        else:
             data_norm = (data * 255.0)
             
        data_uint8 = np.clip(data_norm, 0, 255).astype(np.uint8)
        
        if data_uint8.ndim == 3 and data_uint8.shape[2] == 3:
             # RGB to BGR for OpenCV if it's color
             data_save = cv2.cvtColor(data_uint8, cv2.COLOR_RGB2BGR)
        else:
             data_save = data_uint8
             
        cv2.imwrite(str(jpg_path), data_save)
        return f"Converted {xisf_path.name}"

    except Exception as e:
        return f"Error converting {xisf_path.name}: {e}"

print("\n--- Starting JPEG Conversion (Debayered Images ONLY) ---")
# Ensure base_debayered is defined
if 'base_debayered' not in locals():
    print("Warning: 'base_debayered' not found in context. Assuming './data/debayered'")
    base_debayered = Path(os.getcwd()) / 'data' / 'debayered'

if base_debayered.exists():
    # Find all xisf files recursively in the debayered folder
    all_xisfs = list(base_debayered.rglob('*.xisf'))
    print(f"Found {len(all_xisfs)} debayered XISF files.")
    
    # Execute Conversion
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
        results = list(executor.map(convert_xisf_to_jpg, all_xisfs))
        
    # Summary
    converted = sum(1 for r in results if "Converted" in r)
    skipped = sum(1 for r in results if "Skipped" in r)
    print(f"Done. Converted: {converted}, Skipped: {skipped}")
else:
    print(f"Directory not found: {base_debayered}")


--- Starting JPEG Conversion (Debayered Images ONLY) ---
Found 851 debayered XISF files.
